In [12]:
import pandas as pd

In [13]:
fact_re_state_2025 = pd.read_csv("../clean/fact_re_state_2025.csv")
fact_re_state_potential = pd.read_csv("../clean/fact_re_state_potential.csv")
fact_re_capcity_long = pd.read_csv("../clean/fact_re_capacity_long.csv")

Merge the three cleaned datasets and add new features

In [14]:
mnre_features = fact_re_state_2025[
    [
        "state_id",
        "state_name",
        "Small_Hydro_MW",
        "Wind_Power_MW",
        "Bio_Power_MW",
        "Solar_Power_MW",
        "Large_Hydro_MW",
        "Total_RES_MW"
    ]
].copy()

In [15]:
mnre_features = mnre_features.merge(
    fact_re_state_potential[
        [
            "state_id",
            "total_re_potential_mw"
        ]
    ],
    on="state_id",
    how="left",
    validate="one_to_one"
)

In [16]:
mnre_features["renewable_headroom_mw"] = (
    mnre_features["total_re_potential_mw"]
    - mnre_features["Total_RES_MW"]
)

In [17]:
mnre_features["renewable_utilization_pct"] = (
    mnre_features["Total_RES_MW"]
    / mnre_features["total_re_potential_mw"]
    * 100
)

In [19]:
history_pivot = fact_re_capcity_long.pivot(
    index="state_id",
    columns="fiscal_year",
    values="cumulative_re_capacity_mw"
).reset_index()

In [20]:
history_pivot = history_pivot.rename(columns={
    "2017_18": "re_capacity_2017_18_mw",
    "2024_25": "re_capacity_2024_25_mw"
})

In [21]:
mnre_features = mnre_features.merge(
    history_pivot[
        [
            "state_id",
            "re_capacity_2017_18_mw",
            "re_capacity_2024_25_mw"
        ]
    ],
    on="state_id",
    how="left",
    validate="one_to_one"
)

In [22]:
mnre_features["re_capacity_growth_mw"] = (
    mnre_features["re_capacity_2024_25_mw"]
    - mnre_features["re_capacity_2017_18_mw"]
)

In [23]:
mnre_features["re_capacity_growth_pct"] = (
    mnre_features["re_capacity_growth_mw"]
    / mnre_features["re_capacity_2017_18_mw"]
    * 100
)

In [24]:
mnre_features["re_capacity_cagr_pct"] = (
    (
        mnre_features["re_capacity_2024_25_mw"]
        / mnre_features["re_capacity_2017_18_mw"]
    ) ** (1 / 7)
    - 1
) * 100

In [25]:
print(mnre_features.shape)
print(mnre_features.columns.tolist())
print(mnre_features.isna().sum())
print(
    mnre_features[
        [
            "state_name",
            "Total_RES_MW",
            "total_re_potential_mw",
            "renewable_headroom_mw",
            "renewable_utilization_pct",
            "re_capacity_growth_mw",
            "re_capacity_growth_pct",
            "re_capacity_cagr_pct"
        ]
    ].to_string(index=False)
)

(36, 16)
['state_id', 'state_name', 'Small_Hydro_MW', 'Wind_Power_MW', 'Bio_Power_MW', 'Solar_Power_MW', 'Large_Hydro_MW', 'Total_RES_MW', 'total_re_potential_mw', 'renewable_headroom_mw', 'renewable_utilization_pct', 're_capacity_2017_18_mw', 're_capacity_2024_25_mw', 're_capacity_growth_mw', 're_capacity_growth_pct', 're_capacity_cagr_pct']
state_id                     0
state_name                   0
Small_Hydro_MW               0
Wind_Power_MW                0
Bio_Power_MW                 0
Solar_Power_MW               0
Large_Hydro_MW               0
Total_RES_MW                 0
total_re_potential_mw        0
renewable_headroom_mw        0
renewable_utilization_pct    0
re_capacity_2017_18_mw       1
re_capacity_2024_25_mw       1
re_capacity_growth_mw        1
re_capacity_growth_pct       1
re_capacity_cagr_pct         1
dtype: int64
                              state_name  Total_RES_MW  total_re_potential_mw  renewable_headroom_mw  renewable_utilization_pct  re_capacity_growt

In [26]:
print(
    fact_re_state_potential[
        fact_re_state_potential["state_name"].isin([
            "Chandigarh",
            "Ladakh"
        ])
    ].to_string(index=False)
)

state_id state_name  wind_potential_mw  small_hydro_potential_mw  biomass_potential_mw  bagasse_cogen_potential_mw  solar_ground_potential_mw  large_hydro_potential_mw  total_re_potential_mw
   IN-LA     Ladakh                1.0                       0.0                  0.00                         0.0                    8556.64                       0.0                8557.64
   IN-CH Chandigarh                0.0                       0.0                  0.15                         0.0                      22.42                       0.0                  22.57


In [28]:
import numpy as np

In [29]:
mnre_features[
    ["re_capacity_growth_pct", "re_capacity_cagr_pct"]
] = mnre_features[
    ["re_capacity_growth_pct", "re_capacity_cagr_pct"]
].replace([np.inf, -np.inf], np.nan)

In [30]:
mnre_features = mnre_features.rename(columns={
    "Small_Hydro_MW": "small_hydro_mw",
    "Wind_Power_MW": "wind_power_mw",
    "Bio_Power_MW": "bio_power_mw",
    "Solar_Power_MW": "solar_power_mw",
    "Large_Hydro_MW": "large_hydro_mw",
    "Total_RES_MW": "total_re_mw"
})

In [31]:
print(mnre_features.shape)
print(mnre_features.columns.tolist())
print(mnre_features.isna().sum())

(36, 16)
['state_id', 'state_name', 'small_hydro_mw', 'wind_power_mw', 'bio_power_mw', 'solar_power_mw', 'large_hydro_mw', 'total_re_mw', 'total_re_potential_mw', 'renewable_headroom_mw', 'renewable_utilization_pct', 're_capacity_2017_18_mw', 're_capacity_2024_25_mw', 're_capacity_growth_mw', 're_capacity_growth_pct', 're_capacity_cagr_pct']
state_id                     0
state_name                   0
small_hydro_mw               0
wind_power_mw                0
bio_power_mw                 0
solar_power_mw               0
large_hydro_mw               0
total_re_mw                  0
total_re_potential_mw        0
renewable_headroom_mw        0
renewable_utilization_pct    0
re_capacity_2017_18_mw       1
re_capacity_2024_25_mw       1
re_capacity_growth_mw        1
re_capacity_growth_pct       2
re_capacity_cagr_pct         2
dtype: int64


In [32]:
mnre_features.to_csv(
    "../clean/mnre_features.csv",
    index=False
)

MNRE BIO

In [2]:
import os
import pandas as pd

raw_path = "../raw"

mnre_files = [
    f for f in os.listdir(raw_path)
    if "bio" in f.lower() or "solar" in f.lower()
]

print("=" * 70)
print("MNRE BIO / SOLAR FILES")
print("=" * 70)

for f in mnre_files:
    print(f)

print("\n" + "=" * 70)
print("FILE INSPECTION")
print("=" * 70)

for file in mnre_files:
    path = os.path.join(raw_path, file)
    df = pd.read_csv(path)

    print("\n" + "-" * 70)
    print("FILE:", file)
    print("-" * 70)

    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())

    print("\nDtypes:")
    print(df.dtypes)

    print("\nFirst 5 rows:")
    print(df.head())

    print("\nMissing values:")
    print(df.isna().sum())

    print("\nDuplicate rows:", df.duplicated().sum())

MNRE BIO / SOLAR FILES
mnre_bio_and_solar_breakdown_2025.csv

FILE INSPECTION

----------------------------------------------------------------------
FILE: mnre_bio_and_solar_breakdown_2025.csv
----------------------------------------------------------------------
Shape: (41, 9)
Columns: ['State_UT', 'Biomass_Bagasse_Cogen_MW', 'Biomass_Non_Bagasse_Cogen_MW', 'Waste_to_Energy_Grid_MW', 'Waste_to_Energy_Offgrid_MW', 'Solar_Ground_Mounted_MW', 'Solar_Rooftop_MW', 'Solar_Hybrid_Component_MW', 'Solar_Offgrid_KUSUM_MW']

Dtypes:
State_UT                            str
Biomass_Bagasse_Cogen_MW        float64
Biomass_Non_Bagasse_Cogen_MW    float64
Waste_to_Energy_Grid_MW         float64
Waste_to_Energy_Offgrid_MW      float64
Solar_Ground_Mounted_MW         float64
Solar_Rooftop_MW                float64
Solar_Hybrid_Component_MW       float64
Solar_Offgrid_KUSUM_MW          float64
dtype: object

First 5 rows:
            State_UT  Biomass_Bagasse_Cogen_MW  Biomass_Non_Bagasse_Cogen_MW  \
0

In [4]:
bio_solar = pd.read_csv(
    "../raw/mnre_bio_and_solar_breakdown_2025.csv"
)

print("=" * 70)
print("STATE / UT VALUES")
print("=" * 70)

print(bio_solar["State_UT"].tolist())

print("\n" + "=" * 70)
print("VALUE COUNTS")
print("=" * 70)

print(bio_solar["State_UT"].value_counts())

print("\n" + "=" * 70)
print("TOTALS / NON-STATE ROWS CHECK")
print("=" * 70)

for name in bio_solar["State_UT"]:
    if name.lower() in ["total", "others", "grand total"]:
        print("Possible report row:", repr(name))

STATE / UT VALUES
['Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chhattisgarh', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jammu & Kashmir', 'Jharkhand', 'Karnataka', 'Kerala', 'Ladakh', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Telangana', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal', 'Andaman &', 'Nicobar Islands', 'Chandigarh', 'Dadra & Nagar', 'Haveli /', 'Daman & Diu', 'Delhi', 'Lakshadweep', 'Pondicherry', 'Others', 'Total (MW)']

VALUE COUNTS
State_UT
Andhra Pradesh       1
Arunachal Pradesh    1
Assam                1
Bihar                1
Chhattisgarh         1
Goa                  1
Gujarat              1
Haryana              1
Himachal Pradesh     1
Jammu & Kashmir      1
Jharkhand            1
Karnataka            1
Kerala               1
Ladakh               1
Madhya Pradesh       1
Maharashtra          1
Manipur              1
Meghalaya          

In [6]:
# Remove report-level rows
bio_solar = bio_solar[
    ~bio_solar["State_UT"].isin(["Others", "Total (MW)"])
].copy()

# Reconstruct split state/UT names
bio_solar["State_UT"] = bio_solar["State_UT"].replace({
    "Andaman &": "Andaman and Nicobar Islands",
    "Nicobar Islands": "Andaman and Nicobar Islands",

    "Dadra & Nagar": "Dadra and Nagar Haveli and Daman and Diu",
    "Haveli /": "Dadra and Nagar Haveli and Daman and Diu",
    "Daman & Diu": "Dadra and Nagar Haveli and Daman and Diu",

    "Jammu & Kashmir": "Jammu and Kashmir",
    "Pondicherry": "Puducherry"
})

print("Shape after removing report rows:", bio_solar.shape)

print("\nState names:")
print(bio_solar["State_UT"].tolist())

Shape after removing report rows: (39, 9)

State names:
['Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chhattisgarh', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jammu and Kashmir', 'Jharkhand', 'Karnataka', 'Kerala', 'Ladakh', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Telangana', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal', 'Andaman and Nicobar Islands', 'Andaman and Nicobar Islands', 'Chandigarh', 'Dadra and Nagar Haveli and Daman and Diu', 'Dadra and Nagar Haveli and Daman and Diu', 'Dadra and Nagar Haveli and Daman and Diu', 'Delhi', 'Lakshadweep', 'Puducherry']


In [7]:
bio_cols = [
    "Biomass_Bagasse_Cogen_MW",
    "Biomass_Non_Bagasse_Cogen_MW",
    "Waste_to_Energy_Grid_MW",
    "Waste_to_Energy_Offgrid_MW",
    "Solar_Ground_Mounted_MW",
    "Solar_Rooftop_MW",
    "Solar_Hybrid_Component_MW",
    "Solar_Offgrid_KUSUM_MW"
]

bio_solar = (
    bio_solar
    .groupby("State_UT", as_index=False)[bio_cols]
    .sum()
)

print("Shape:", bio_solar.shape)

print("\nDuplicate states:")
print(bio_solar["State_UT"].duplicated().sum())

print("\nStates:", bio_solar["State_UT"].tolist())

Shape: (36, 9)

Duplicate states:
0

States: ['Andaman and Nicobar Islands', 'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chandigarh', 'Chhattisgarh', 'Dadra and Nagar Haveli and Daman and Diu', 'Delhi', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jammu and Kashmir', 'Jharkhand', 'Karnataka', 'Kerala', 'Ladakh', 'Lakshadweep', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Puducherry', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Telangana', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal']


In [8]:
bio_solar

,State_UT,Biomass_Bagasse_Cogen_MW,Biomass_Non_Bagasse_Cogen_MW,Waste_to_Energy_Grid_MW,Waste_to_Energy_Offgrid_MW,Solar_Ground_Mounted_MW,Solar_Rooftop_MW,Solar_Hybrid_Component_MW,Solar_Offgrid_KUSUM_MW
0,Andaman and Nicobar Islands,0.00,0.00,0.00,0.00,25.05,4.59,0.0,0.27
1,Andhra Pradesh,378.10,113.57,53.16,49.19,4990.86,290.80,0.0,88.34
2,Arunachal Pradesh,0.00,0.00,0.00,0.00,1.27,6.68,0.0,6.90
3,Assam,0.00,2.00,0.00,0.00,126.00,61.10,0.0,9.44
4,Bihar,112.50,26.40,0.00,1.32,196.06,111.00,0.0,21.28
5,Chandigarh,0.00,0.00,0.00,0.00,6.34,71.70,0.0,0.81
6,Chhattisgarh,272.09,2.50,0.00,10.83,848.91,107.40,0.0,390.73
7,Dadra and Nagar Haveli and Daman and Diu,3.75,0.00,0.00,0.00,14.30,33.82,0.0,0.00
8,Delhi,0.00,0.00,84.00,0.00,9.84,302.10,0.0,1.46
9,Goa,0.00,0.00,1.94,0.00,1.95,53.00,0.0,1.49


In [10]:
# ------------------------------------------------------------
# 1. Rename state column
# ------------------------------------------------------------

bio_solar = bio_solar.rename(
    columns={"State_UT": "state_name"}
)
dim_state = pd.read_csv("../clean/dim_state.csv")
# ------------------------------------------------------------
# 2. Attach canonical state_id
# ------------------------------------------------------------

bio_solar = bio_solar.merge(
    dim_state[["state_id", "state_name"]],
    on="state_name",
    how="left",
    validate="one_to_one"
)

print("Shape:", bio_solar.shape)
print("Missing state IDs:", bio_solar["state_id"].isna().sum())

# ------------------------------------------------------------
# 3. Reconciliation totals
# ------------------------------------------------------------

bio_solar["biomass_breakdown_total_mw"] = (
    bio_solar["Biomass_Bagasse_Cogen_MW"]
    + bio_solar["Biomass_Non_Bagasse_Cogen_MW"]
)

bio_solar["waste_to_energy_total_mw"] = (
    bio_solar["Waste_to_Energy_Grid_MW"]
    + bio_solar["Waste_to_Energy_Offgrid_MW"]
)

bio_solar["solar_breakdown_total_mw"] = (
    bio_solar["Solar_Ground_Mounted_MW"]
    + bio_solar["Solar_Rooftop_MW"]
    + bio_solar["Solar_Hybrid_Component_MW"]
    + bio_solar["Solar_Offgrid_KUSUM_MW"]
)

# ------------------------------------------------------------
# 4. Inspect totals
# ------------------------------------------------------------

print("\nBreakdown totals:")
print(
    bio_solar[
        [
            "state_name",
            "biomass_breakdown_total_mw",
            "waste_to_energy_total_mw",
            "solar_breakdown_total_mw"
        ]
    ].to_string(index=False)
)

Shape: (36, 10)
Missing state IDs: 0

Breakdown totals:
                              state_name  biomass_breakdown_total_mw  waste_to_energy_total_mw  solar_breakdown_total_mw
             Andaman and Nicobar Islands                        0.00                      0.00                     29.91
                          Andhra Pradesh                      491.67                    102.35                   5370.00
                       Arunachal Pradesh                        0.00                      0.00                     14.85
                                   Assam                        2.00                      0.00                    196.54
                                   Bihar                      138.90                      1.32                    328.34
                              Chandigarh                        0.00                      0.00                     78.85
                            Chhattisgarh                      274.59                     10.83   

In [11]:
bio_solar_final = bio_solar.rename(columns={
    "Biomass_Bagasse_Cogen_MW": "biomass_bagasse_cogen_mw",
    "Biomass_Non_Bagasse_Cogen_MW": "biomass_non_bagasse_cogen_mw",
    "Waste_to_Energy_Grid_MW": "waste_to_energy_grid_mw",
    "Waste_to_Energy_Offgrid_MW": "waste_to_energy_offgrid_mw",
    "Solar_Ground_Mounted_MW": "solar_ground_mounted_mw",
    "Solar_Rooftop_MW": "solar_rooftop_mw",
    "Solar_Hybrid_Component_MW": "solar_hybrid_component_mw",
    "Solar_Offgrid_KUSUM_MW": "solar_offgrid_kusum_mw"
})

bio_solar_final = bio_solar_final[
    [
        "state_id",
        "state_name",
        "biomass_bagasse_cogen_mw",
        "biomass_non_bagasse_cogen_mw",
        "waste_to_energy_grid_mw",
        "waste_to_energy_offgrid_mw",
        "solar_ground_mounted_mw",
        "solar_rooftop_mw",
        "solar_hybrid_component_mw",
        "solar_offgrid_kusum_mw",
        "biomass_breakdown_total_mw",
        "waste_to_energy_total_mw",
        "solar_breakdown_total_mw"
    ]
]

print("Shape:", bio_solar_final.shape)
print("Unique state IDs:", bio_solar_final["state_id"].nunique())
print("\nMissing values:")
print(bio_solar_final.isna().sum())

Shape: (36, 13)
Unique state IDs: 36

Missing values:
state_id                        0
state_name                      0
biomass_bagasse_cogen_mw        0
biomass_non_bagasse_cogen_mw    0
waste_to_energy_grid_mw         0
waste_to_energy_offgrid_mw      0
solar_ground_mounted_mw         0
solar_rooftop_mw                0
solar_hybrid_component_mw       0
solar_offgrid_kusum_mw          0
biomass_breakdown_total_mw      0
waste_to_energy_total_mw        0
solar_breakdown_total_mw        0
dtype: int64


In [12]:
bio_solar_final

,state_id,state_name,biomass_bagasse_cogen_mw,biomass_non_bagasse_cogen_mw,waste_to_energy_grid_mw,waste_to_energy_offgrid_mw,solar_ground_mounted_mw,solar_rooftop_mw,solar_hybrid_component_mw,solar_offgrid_kusum_mw,biomass_breakdown_total_mw,waste_to_energy_total_mw,solar_breakdown_total_mw
0,IN-AN,Andaman and Nicobar Islands,0.00,0.00,0.00,0.00,25.05,4.59,0.0,0.27,0.00,0.00,29.91
1,IN-AP,Andhra Pradesh,378.10,113.57,53.16,49.19,4990.86,290.80,0.0,88.34,491.67,102.35,5370.00
2,IN-AR,Arunachal Pradesh,0.00,0.00,0.00,0.00,1.27,6.68,0.0,6.90,0.00,0.00,14.85
3,IN-AS,Assam,0.00,2.00,0.00,0.00,126.00,61.10,0.0,9.44,2.00,0.00,196.54
4,IN-BR,Bihar,112.50,26.40,0.00,1.32,196.06,111.00,0.0,21.28,138.90,1.32,328.34
5,IN-CH,Chandigarh,0.00,0.00,0.00,0.00,6.34,71.70,0.0,0.81,0.00,0.00,78.85
6,IN-CT,Chhattisgarh,272.09,2.50,0.00,10.83,848.91,107.40,0.0,390.73,274.59,10.83,1347.04
7,IN-DH,Dadra and Nagar Haveli and Daman and Diu,3.75,0.00,0.00,0.00,14.30,33.82,0.0,0.00,3.75,0.00,48.12
8,IN-DL,Delhi,0.00,0.00,84.00,0.00,9.84,302.10,0.0,1.46,0.00,84.00,313.40
9,IN-GA,Goa,0.00,0.00,1.94,0.00,1.95,53.00,0.0,1.49,0.00,1.94,56.44


In [13]:
bio_solar_final.to_csv(
    "../clean/fact_re_bio_solar_breakdown_2025.csv",
    index=False,
    encoding="utf-8"
)

print("Saved successfully.")
print("Shape:", bio_solar_final.shape)
print("Path: ../data/clean/fact_re_bio_solar_breakdown_2025.csv")

Saved successfully.
Shape: (36, 13)
Path: ../data/clean/fact_re_bio_solar_breakdown_2025.csv
